# Notebook 2: Métricas de Evaluación para Clasificación
## Diplomado en Machine Learning para Seguros · Subtema 3

---

### ¿Qué vamos a hacer en este notebook?

Trabajamos con el **mismo ejemplo de la presentación**: detección de fraude en seguros de auto.

**La base de datos:**
- 10,000 reclamaciones de seguros de auto
- 400 son fraudes reales (4% de prevalencia)
- El modelo GBM ha sido entrenado con 40,000 reclamaciones históricas

**Variables del asegurado y la reclamación:**
| Variable | Descripción |
|----------|-------------|
| `monto_reclamacion` | Monto total reclamado en pesos |
| `dias_reporte` | Días entre el siniestro y el reporte |
| `num_testigos` | Número de testigos declarados |
| `hora_siniestro` | Hora del siniestro (0-23) |
| `anios_poliza` | Antigüedad de la póliza en años |
| `zona_riesgo` | Zona geográfica de riesgo (1-5) |
| `fraude` | **Variable objetivo**: 1=fraude, 0=legítimo |

---
**Referencia en la presentación:** Parte 2, slides 3.0 al 3.9

## Sección 0: Importar librerías

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score,
    precision_recall_curve, average_precision_score
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


---
## Sección 1: Generar y explorar la base de datos

Simulamos 50,000 reclamaciones históricas (2019-2022) para entrenamiento y 10,000 de 2023 para evaluación.

In [14]:
def generar_reclamaciones(n, fraude_rate=0.04, seed=None):
    """
    Genera una base de datos de reclamaciones de seguros de auto.
    El fraude es más probable cuando:
    - El monto es muy alto
    - Hay pocos o ningún testigo
    - El reporte tarda muchos días
    - El siniestro ocurre de noche
    """
    if seed is not None:
        np.random.seed(seed)

    # Características de la reclamación
    monto          = np.random.lognormal(10.5, 1.2, n)          # montos con cola pesada
    dias_reporte   = np.clip(np.random.exponential(4, n), 0, 60).astype(int)
    num_testigos   = np.random.choice([0, 1, 2, 3], n, p=[0.35, 0.40, 0.18, 0.07])
    hora_siniestro = np.random.randint(0, 24, n)
    anios_poliza   = np.clip(np.random.exponential(3, n), 0, 15).round(1)
    zona_riesgo    = np.random.randint(1, 6, n)

    # Probabilidad de fraude: función logística de las variables
    log_odds = (
        -3.5                                                          # intercepto (base baja)
        + 0.4 * (monto > np.percentile(monto, 80)).astype(float)     # montos altos
        + 0.6 * (num_testigos == 0).astype(float)                    # sin testigos
        + 0.35 * (dias_reporte > 10).astype(float)                   # reporte tardío
        + 0.3 * (hora_siniestro >= 22).astype(float)                 # de noche
        + 0.2 * (zona_riesgo >= 4).astype(float)                     # zona de alto riesgo
    )
    prob_fraude = 1 / (1 + np.exp(-log_odds))
    fraude = (np.random.rand(n) < prob_fraude).astype(int)

    return pd.DataFrame({
        'monto_reclamacion': monto.round(2),
        'dias_reporte': dias_reporte,
        'num_testigos': num_testigos,
        'hora_siniestro': hora_siniestro,
        'anios_poliza': anios_poliza,
        'zona_riesgo': zona_riesgo,
        'fraude': fraude
    })

# Generar datos históricos (entrenamiento) y datos 2023 (evaluación)
print("Generando datos...")
df_train_hist = generar_reclamaciones(40000, seed=54)   # 2019-2022
df_eval_2023  = generar_reclamaciones(10000, seed=99)   # 2023

print(f"\n{'='*55}")
print(f"  RESUMEN DE LAS BASES DE DATOS")
print(f"{'='*55}")
for nombre, df_tmp in [('Histórico 2019-2022 (TRAIN)', df_train_hist),
                        ('Evaluación 2023 (TEST)',       df_eval_2023)]:
    n_fraude = df_tmp['fraude'].sum()
    print(f"\n  {nombre}:")
    print(f"    Total reclamaciones: {len(df_tmp):,}")
    print(f"    Fraudes reales:      {n_fraude:,} ({n_fraude/len(df_tmp)*100:.1f}%)")
    print(f"    Legítimas:           {len(df_tmp)-n_fraude:,} ({(len(df_tmp)-n_fraude)/len(df_tmp)*100:.1f}%)")
    print(f"    Monto promedio:      ${df_tmp['monto_reclamacion'].mean():,.0f}")
    print(f"    Días reporte prom.:  {df_tmp['dias_reporte'].mean():.1f} días")

print(f"\n  ⚠️ Desbalance evidente: {df_eval_2023['fraude'].mean()*100:.1f}% fraude vs {(1-df_eval_2023['fraude'].mean())*100:.1f}% legítimas")
print(f"  Un modelo trivial que diga 'todo es legítimo' tendría accuracy = {(1-df_eval_2023['fraude'].mean())*100:.1f}%")

Generando datos...

  RESUMEN DE LAS BASES DE DATOS

  Histórico 2019-2022 (TRAIN):
    Total reclamaciones: 40,000
    Fraudes reales:      1,923 (4.8%)
    Legítimas:           38,077 (95.2%)
    Monto promedio:      $74,471
    Días reporte prom.:  3.5 días

  Evaluación 2023 (TEST):
    Total reclamaciones: 10,000
    Fraudes reales:      469 (4.7%)
    Legítimas:           9,531 (95.3%)
    Monto promedio:      $77,638
    Días reporte prom.:  3.6 días

  ⚠️ Desbalance evidente: 4.7% fraude vs 95.3% legítimas
  Un modelo trivial que diga 'todo es legítimo' tendría accuracy = 95.3%


---
## Sección 2: Entrenar el modelo GBM

El modelo se entrena con datos históricos y se evalúa sobre las reclamaciones de 2023.

In [15]:
# Variables predictoras
features = ['monto_reclamacion', 'dias_reporte', 'num_testigos',
            'hora_siniestro', 'anios_poliza', 'zona_riesgo']

X_hist  = df_train_hist[features].values
y_hist  = df_train_hist['fraude'].values
X_2023  = df_eval_2023[features].values
y_2023  = df_eval_2023['fraude'].values

# Escalamiento: fit SOLO en histórico
scaler = StandardScaler()
X_hist_sc  = scaler.fit_transform(X_hist)
X_2023_sc  = scaler.transform(X_2023)     # ← solo transform, NO fit

# Entrenar GBM con datos históricos
print("Entrenando GBM con 40,000 reclamaciones históricas (2019-2022)...")
gbm = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)
gbm.fit(X_hist_sc, y_hist)

# Predicciones sobre 2023
y_pred_clase = gbm.predict(X_2023_sc)              # etiqueta: 0 o 1
y_pred_prob  = gbm.predict_proba(X_2023_sc)[:, 1]  # probabilidad de fraude

print("✅ Modelo entrenado.")
print(f"\nPredicciones sobre 10,000 reclamaciones de 2023:")
print(f"  Marcadas como fraude: {y_pred_clase.sum():,} ({y_pred_clase.mean()*100:.1f}%)")
print(f"  Marcadas como legítimas: {(1-y_pred_clase).sum():,} ({(1-y_pred_clase).mean()*100:.1f}%)")

Entrenando GBM con 40,000 reclamaciones históricas (2019-2022)...
✅ Modelo entrenado.

Predicciones sobre 10,000 reclamaciones de 2023:
  Marcadas como fraude: 10 (0.1%)
  Marcadas como legítimas: 9,990 (99.9%)


---
## Sección 3: La Matriz de Confusión

**Todas las métricas de clasificación se derivan de estos 4 números:** VP, FN, FP, VN.

In [16]:
# Calcular la matriz de confusión
cm = confusion_matrix(y_2023, y_pred_clase)
vn, fp, fn, vp = cm.ravel()   # tn=VN, fp=FP, fn=FN, tp=VP
n_total = len(y_2023)

print("=" * 65)
print("  MATRIZ DE CONFUSIÓN — MODELO DE FRAUDE")
print("=" * 65)
print()
print("                    PREDICHO")
print("                 FRAUDE    LEGÍTIMO")
print(f"  REAL  FRAUDE    VP={vp:>4}    FN={fn:>4}")
print(f"        LEGÍTIMO  FP={fp:>4}    VN={vn:>4}")
print()

# Interpretación económica (de la Slide 3.1)
FRAUDE_PROMEDIO   = 35000   # $ por fraude
COSTO_INV_SIU     = 3500    # $ por investigación del SIU

valor_vp = vp * FRAUDE_PROMEDIO
costo_fn = fn * FRAUDE_PROMEDIO
costo_fp = fp * COSTO_INV_SIU
valor_neto = valor_vp - costo_fn - costo_fp
costo_sin_modelo = (vp + fn) * FRAUDE_PROMEDIO  # si no detectáramos nada

print("  ANÁLISIS ECONÓMICO:")
print(f"  {'Celda':25s}  {'Cant.':>6}  {'Impacto económico':>22}")
print(f"  {'-'*60}")
print(f"  {'VP — Fraudes detectados':25s}  {vp:>6}  ${valor_vp:>20,.0f} evitados ✅")
print(f"  {'FN — Fraudes perdidos':25s}  {fn:>6}  ${costo_fn:>20,.0f} pagados ❌")
print(f"  {'FP — Falsas alarmas':25s}  {fp:>6}  ${costo_fp:>20,.0f} investigación ⚠️")
print(f"  {'VN — Legítimos correctos':25s}  {vn:>6}  {'Sin impacto adicional':>22}")
print(f"  {'-'*60}")
print(f"  {'VALOR NETO DEL MODELO':25s}         ${valor_neto:>20,.0f}")
print(f"  {'Sin modelo (cero detección)':25s}         ${-costo_sin_modelo:>20,.0f}")
print(f"  {'Mejora vs. no hacer nada':25s}         ${valor_neto + costo_sin_modelo:>20,.0f}")

  MATRIZ DE CONFUSIÓN — MODELO DE FRAUDE

                    PREDICHO
                 FRAUDE    LEGÍTIMO
  REAL  FRAUDE    VP=   2    FN= 467
        LEGÍTIMO  FP=   8    VN=9523

  ANÁLISIS ECONÓMICO:
  Celda                       Cant.       Impacto económico
  ------------------------------------------------------------
  VP — Fraudes detectados         2  $              70,000 evitados ✅
  FN — Fraudes perdidos         467  $          16,345,000 pagados ❌
  FP — Falsas alarmas             8  $              28,000 investigación ⚠️
  VN — Legítimos correctos     9523   Sin impacto adicional
  ------------------------------------------------------------
  VALOR NETO DEL MODELO             $         -16,303,000
  Sin modelo (cero detección)         $         -16,415,000
  Mejora vs. no hacer nada          $             112,000


---
## Sección 4: Accuracy, Precisión, Recall y Especificidad

Calculamos las 4 métricas paso a paso y explicamos cuándo usar cada una.

In [17]:
print("=" * 70)
print("  LAS 4 MÉTRICAS BÁSICAS — CÁLCULO EXPLÍCITO")
print("=" * 70)

# 1. ACCURACY
accuracy = (vp + vn) / n_total
# ¿Qué daría el modelo trivial "todo es legítimo"?
accuracy_trivial = vn / n_total   # solo acierta en los negativos
print(f"\n1. ACCURACY = (VP + VN) / N")
print(f"   = ({vp} + {vn:,}) / {n_total:,} = {accuracy*100:.2f}%")
print(f"   Modelo trivial 'todo es legítimo': {accuracy_trivial*100:.2f}%")
print(f"   Mejora real de nuestro modelo: {(accuracy-accuracy_trivial)*100:.2f} pp")
print(f"   ⚠️ CON 4% DE FRAUDE, LA ACCURACY ES COMPLETAMENTE ENGAÑOSA")

# 2. PRECISIÓN
precision = vp / (vp + fp) if (vp + fp) > 0 else 0
precision_sk = precision_score(y_2023, y_pred_clase)
print(f"\n2. PRECISIÓN = VP / (VP + FP)")
print(f"   = {vp} / ({vp} + {fp}) = {vp} / {vp+fp} = {precision*100:.2f}%")
print(f"   De las {vp+fp:,} alarmas generadas, {vp} ({precision*100:.1f}%) son fraudes reales")
print(f"   {fp} ({fp/(vp+fp)*100:.1f}%) son falsas alarmas (costo: ${fp*COSTO_INV_SIU:,.0f})")
print(f"   ✅ sklearn: {precision_sk*100:.2f}%")

# 3. RECALL (Sensibilidad)
recall = vp / (vp + fn) if (vp + fn) > 0 else 0
recall_sk = recall_score(y_2023, y_pred_clase)
print(f"\n3. RECALL = VP / (VP + FN)")
print(f"   = {vp} / ({vp} + {fn}) = {vp} / {vp+fn} = {recall*100:.2f}%")
print(f"   De los {vp+fn:,} fraudes reales, detectamos {vp} ({recall*100:.1f}%)")
print(f"   {fn} fraudes se escaparon (pérdida: ${fn*FRAUDE_PROMEDIO:,.0f})")
print(f"   ✅ sklearn: {recall_sk*100:.2f}%")

# 4. ESPECIFICIDAD
especificidad = vn / (vn + fp) if (vn + fp) > 0 else 0
print(f"\n4. ESPECIFICIDAD = VN / (VN + FP)")
print(f"   = {vn:,} / ({vn:,} + {fp}) = {especificidad*100:.2f}%")
print(f"   Casi todos los legítimos se clasifican bien")
print(f"   Alta en datos desbalanceados — no es muy informativa para fraude")

print(f"\n  CUÁNDO PRIORIZAR CADA MÉTRICA:")
print(f"  Precisión  → cuando FP son costosos (cancelar pólizas automáticamente, rechazar reclamaciones)")
print(f"  Recall     → cuando FN son costosos (fraude: ${FRAUDE_PROMEDIO:,} por caso perdido)")
print(f"  Ambas      → cuando necesitas balance → usar F1")

  LAS 4 MÉTRICAS BÁSICAS — CÁLCULO EXPLÍCITO

1. ACCURACY = (VP + VN) / N
   = (2 + 9,523) / 10,000 = 95.25%
   Modelo trivial 'todo es legítimo': 95.23%
   Mejora real de nuestro modelo: 0.02 pp
   ⚠️ CON 4% DE FRAUDE, LA ACCURACY ES COMPLETAMENTE ENGAÑOSA

2. PRECISIÓN = VP / (VP + FP)
   = 2 / (2 + 8) = 2 / 10 = 20.00%
   De las 10 alarmas generadas, 2 (20.0%) son fraudes reales
   8 (80.0%) son falsas alarmas (costo: $28,000)
   ✅ sklearn: 20.00%

3. RECALL = VP / (VP + FN)
   = 2 / (2 + 467) = 2 / 469 = 0.43%
   De los 469 fraudes reales, detectamos 2 (0.4%)
   467 fraudes se escaparon (pérdida: $16,345,000)
   ✅ sklearn: 0.43%

4. ESPECIFICIDAD = VN / (VN + FP)
   = 9,523 / (9,523 + 8) = 99.92%
   Casi todos los legítimos se clasifican bien
   Alta en datos desbalanceados — no es muy informativa para fraude

  CUÁNDO PRIORIZAR CADA MÉTRICA:
  Precisión  → cuando FP son costosos (cancelar pólizas automáticamente, rechazar reclamaciones)
  Recall     → cuando FN son costosos (fraud

---
## Sección 5: F1-Score y la Familia Fβ

La media armónica penaliza el valor más bajo de Precisión o Recall.

In [20]:
print("=" * 65)
print("  F1-SCORE Y FAMILIA Fβ")
print("=" * 65)

# F1: media armónica de Precisión y Recall
f1_manual = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
f1_sk = f1_score(y_2023, y_pred_clase)

print(f"\nF1 = 2 × (Precisión × Recall) / (Precisión + Recall)")
print(f"   = 2 × ({precision:.4f} × {recall:.4f}) / ({precision:.4f} + {recall:.4f})")
print(f"   = 2 × {precision*recall:.4f} / {precision+recall:.4f}")
print(f"   = {f1_manual:.4f} = {f1_manual*100:.1f}%")
print(f"   ✅ sklearn: {f1_sk*100:.1f}%")

# Demostrar por qué media armónica y no aritmética
print(f"\n¿POR QUÉ MEDIA ARMÓNICA Y NO ARITMÉTICA?")
print(f"  Media aritmética de {precision*100:.1f}% y {recall*100:.1f}% = {(precision+recall)/2*100:.1f}%")
print(f"  Media armónica (F1) = {f1_manual*100:.1f}%")
print(f"  \nEjemplo extremo: Precisión=90%, Recall=10%")
p_ej, r_ej = 0.90, 0.10
print(f"    Media aritmética = {(p_ej+r_ej)/2*100:.0f}% (parece aceptable)")
print(f"    Media armónica F1 = {2*p_ej*r_ej/(p_ej+r_ej)*100:.1f}% (refleja que el recall es pésimo)")

# Familia Fβ
print(f"\nFAMILIA Fβ: ponderar Recall sobre Precisión (o viceversa)")
from sklearn.metrics import fbeta_score
for beta, desc, uso in [
    (0.5, 'F0.5 — Peso doble a Precisión', 'Cancelación automática de pólizas, rechazo de reclamaciones'),
    (1.0, 'F1  — Peso igual',              'Sin preferencia clara entre FP y FN — el más reportado'),
    (2.0, 'F2  — Peso doble a Recall',     f'Detección de fraude: FN=${FRAUDE_PROMEDIO:,} >> FP=${COSTO_INV_SIU:,}'),
]:
    fb = fbeta_score(y_2023, y_pred_clase, beta=beta)
    print(f"  {desc}: F{beta} = {fb*100:.1f}%  |  {uso}")

print(f"\n  Para detección de fraude: ratio FN/FP = ${FRAUDE_PROMEDIO:,}/${COSTO_INV_SIU:,} = {FRAUDE_PROMEDIO/COSTO_INV_SIU:.0f}x")
print(f"  → Usar F2 para darle el doble de importancia al Recall")

  F1-SCORE Y FAMILIA Fβ

F1 = 2 × (Precisión × Recall) / (Precisión + Recall)
   = 2 × (0.2000 × 0.0043) / (0.2000 + 0.0043)
   = 2 × 0.0009 / 0.2043
   = 0.0084 = 0.8%
   ✅ sklearn: 0.8%

¿POR QUÉ MEDIA ARMÓNICA Y NO ARITMÉTICA?
  Media aritmética de 20.0% y 0.4% = 10.2%
  Media armónica (F1) = 0.8%
  
Ejemplo extremo: Precisión=90%, Recall=10%
    Media aritmética = 50% (parece aceptable)
    Media armónica F1 = 18.0% (refleja que el recall es pésimo)

FAMILIA Fβ: ponderar Recall sobre Precisión (o viceversa)
  F0.5 — Peso doble a Precisión: F0.5 = 2.0%  |  Cancelación automática de pólizas, rechazo de reclamaciones
  F1  — Peso igual: F1.0 = 0.8%  |  Sin preferencia clara entre FP y FN — el más reportado
  F2  — Peso doble a Recall: F2.0 = 0.5%  |  Detección de fraude: FN=$35,000 >> FP=$3,500

  Para detección de fraude: ratio FN/FP = $35,000/$3,500 = 10x
  → Usar F2 para darle el doble de importancia al Recall


---
## Sección 6: Efecto del Umbral de Decisión

sklearn usa τ=0.5 por defecto, pero este umbral es **completamente arbitrario**.

In [21]:
print("EFECTO DEL UMBRAL DE DECISIÓN τ")
print("=" * 80)
print()
print(f"{'τ':>6}  {'Alarmas':>8}  {'VP':>6}  {'FP':>6}  {'Precisión':>10}  {'Recall':>8}  {'F1':>8}  {'J=TPR-FPR':>10}")
print("-" * 80)

n_fraude_total = y_2023.sum()
n_legitimo_total = len(y_2023) - n_fraude_total

resultados_umbral = []
for tau in [0.80, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10]:
    y_pred_tau = (y_pred_prob >= tau).astype(int)
    cm_tau = confusion_matrix(y_2023, y_pred_tau)
    vn_t, fp_t, fn_t, vp_t = cm_tau.ravel()

    prec_t  = vp_t / (vp_t + fp_t) if (vp_t + fp_t) > 0 else 0
    rec_t   = vp_t / (vp_t + fn_t) if (vp_t + fn_t) > 0 else 0
    f1_t    = 2 * prec_t * rec_t / (prec_t + rec_t) if (prec_t + rec_t) > 0 else 0
    fpr_t   = fp_t / n_legitimo_total
    tpr_t   = rec_t
    J_t     = tpr_t - fpr_t
    alarmas = y_pred_tau.sum()

    resultados_umbral.append({'tau': tau, 'VP': vp_t, 'FP': fp_t,
                              'prec': prec_t, 'rec': rec_t, 'f1': f1_t, 'J': J_t})

    marcador = " ← default" if tau == 0.50 else ""
    print(f"  {tau:.2f}  {alarmas:>8,}  {vp_t:>6}  {fp_t:>6}  "
          f"{prec_t*100:>9.1f}%  {rec_t*100:>7.1f}%  {f1_t*100:>7.1f}%  {J_t:>10.3f}{marcador}")

# Umbral de Youden
best_J = max(resultados_umbral, key=lambda x: x['J'])
print()
print(f"  Umbral de Youden (max J = TPR − FPR): τ = {best_J['tau']:.2f}")
print(f"  → Maximiza simultáneamente el recall y minimiza las falsas alarmas")

# Umbral basado en costos
tau_costos = COSTO_INV_SIU / (COSTO_INV_SIU + FRAUDE_PROMEDIO)
print(f"\n  Umbral basado en costos: τ* = C_FP / (C_FP + C_FN)")
print(f"    = ${COSTO_INV_SIU:,} / (${COSTO_INV_SIU:,} + ${FRAUDE_PROMEDIO:,}) = {tau_costos:.3f}")
print(f"  → Con este umbral, investigamos si la probabilidad de fraude > {tau_costos*100:.1f}%")

EFECTO DEL UMBRAL DE DECISIÓN τ

     τ   Alarmas      VP      FP   Precisión    Recall        F1   J=TPR-FPR
--------------------------------------------------------------------------------
  0.80         0       0       0        0.0%      0.0%      0.0%       0.000
  0.60         4       1       3       25.0%      0.2%      0.4%       0.002
  0.50        10       2       8       20.0%      0.4%      0.8%       0.003 ← default
  0.40        14       3      11       21.4%      0.6%      1.2%       0.005
  0.30        25       4      21       16.0%      0.9%      1.6%       0.006
  0.20        65       6      59        9.2%      1.3%      2.2%       0.007
  0.10       368      24     344        6.5%      5.1%      5.7%       0.015

  Umbral de Youden (max J = TPR − FPR): τ = 0.10
  → Maximiza simultáneamente el recall y minimiza las falsas alarmas

  Umbral basado en costos: τ* = C_FP / (C_FP + C_FN)
    = $3,500 / ($3,500 + $35,000) = 0.091
  → Con este umbral, investigamos si la proba

---
## Sección 7: Curva ROC y AUC

In [ ]:
# Calcular la curva ROC
fpr_vals, tpr_vals, thresholds = roc_curve(y_2023, y_pred_prob)
auc = roc_auc_score(y_2023, y_pred_prob)
gini = 2 * auc - 1

# Umbral óptimo de Youden
J_scores = tpr_vals - fpr_vals
idx_youden = J_scores.argmax()
tau_youden = thresholds[idx_youden]

print(f"AUC = {auc:.4f}")
print(f"Gini = 2 × AUC − 1 = 2 × {auc:.4f} − 1 = {gini:.4f}")
print(f"Umbral óptimo de Youden: τ = {tau_youden:.3f} (J = {J_scores[idx_youden]:.3f})")

# Tabla de puntos de la curva (como en la presentación)
print(f"\n  Puntos clave de la curva ROC:")
print(f"  {'τ':>6}  {'TPR (Recall)':>13}  {'FPR':>8}  {'Punto en curva'}")
print(f"  {'-'*55}")
for tau_punto in [1.0, 0.80, 0.50, 0.30, 0.10, 0.0]:
    # Encontrar el punto más cercano en la curva
    idx = np.argmin(np.abs(thresholds - tau_punto)) if tau_punto > 0 else -1
    if tau_punto == 0:
        tpr_p, fpr_p = 1.0, 1.0
    elif tau_punto == 1.0:
        tpr_p, fpr_p = 0.0, 0.0
    else:
        tpr_p = tpr_vals[idx]
        fpr_p = fpr_vals[idx]
    print(f"  {tau_punto:>6.2f}  {tpr_p*100:>12.1f}%  {fpr_p*100:>7.1f}%  ({fpr_p:.3f}, {tpr_p:.3f})")

# Tabla de referencia AUC/Gini
print(f"\n  TABLA DE REFERENCIA AUC / GINI:")
tabla_auc = [
    ('0.90 – 1.00', '0.80 – 1.00', 'Excelente',     'Scoring crediticio maduro — AUC=1.0 → revisar leakage'),
    ('0.80 – 0.89', '0.60 – 0.78', 'Bueno',          'Modelo de fraude bien calibrado, underwriting de vida'),
    ('0.70 – 0.79', '0.40 – 0.58', 'Aceptable',      'Modelos de churn, siniestralidad por zona'),
    ('0.60 – 0.69', '0.20 – 0.38', 'Pobre',          'Agregar más variables predictoras'),
    ('0.50',        '0.00',        'Aleatorio',       'Igual que una moneda — sin capacidad discriminante'),
]
print(f"  {'AUC':12s}  {'Gini':10s}  {'Calidad':12s}  {'Contexto en seguros'}")
print(f"  {'-'*75}")
for row in tabla_auc:
    marcador = " ← NUESTRO MODELO" if '0.80 – 0.89' == row[0] and auc > 0.80 else ""
    print(f"  {row[0]:12s}  {row[1]:10s}  {row[2]:12s}  {row[3]}{marcador}")

print(f"\n  Nuestro modelo: AUC = {auc:.4f}, Gini = {gini:.4f} → ", end="")
if auc > 0.80:
    print("Clasificación BUENA ✅")
elif auc > 0.70:
    print("Clasificación ACEPTABLE ⚠️")
else:
    print("Clasificación POBRE 🔴")

---
## Sección 8: Visualización completa

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Evaluación del Modelo de Detección de Fraude — 10,000 Reclamaciones 2023',
             fontsize=13, fontweight='bold')

# --- 1. Curva ROC ---
ax = axes[0][0]
ax.plot(fpr_vals, tpr_vals, color='#5B21B6', linewidth=2.5,
        label=f'GBM (AUC = {auc:.3f}, Gini = {gini:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Clasificador aleatorio (AUC=0.5)')
ax.fill_between(fpr_vals, tpr_vals, alpha=0.12, color='#7C3AED')
ax.scatter(fpr_vals[idx_youden], tpr_vals[idx_youden],
           color='red', s=120, zorder=5,
           label=f'Youden óptimo (τ={tau_youden:.2f}, J={J_scores[idx_youden]:.3f})')
ax.set_xlabel('FPR (Tasa de Falsos Positivos)')
ax.set_ylabel('TPR (Recall)')
ax.set_title('Curva ROC', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- 2. Matriz de confusión visual ---
ax = axes[0][1]
cm_norm = cm.astype(float)
im = ax.imshow(cm, cmap='Purples', aspect='auto')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Legítimo (0)', 'Fraude (1)'], fontsize=11)
ax.set_yticklabels(['Legítimo (0)', 'Fraude (1)'], fontsize=11)
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title(f'Matriz de Confusión\nPrecisión={precision*100:.1f}%  Recall={recall*100:.1f}%  F1={f1_manual*100:.1f}%',
             fontweight='bold')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black',
                fontsize=16, fontweight='bold')
# Etiquetas VP/FP/FN/VN
labels = [['VN', 'FP'], ['FN', 'VP']]
for i in range(2):
    for j in range(2):
        ax.text(j, i-0.35, labels[i][j],
                ha='center', va='center',
                color='gray', fontsize=10)

# --- 3. Precisión-Recall por umbral ---
ax = axes[1][0]
taus_plot = np.arange(0.01, 0.99, 0.01)
prec_por_tau = []
rec_por_tau  = []
for tau_p in taus_plot:
    y_t = (y_pred_prob >= tau_p).astype(int)
    prec_por_tau.append(precision_score(y_2023, y_t, zero_division=0))
    rec_por_tau.append(recall_score(y_2023, y_t))
ax.plot(taus_plot, prec_por_tau, color='#0D7490', linewidth=2, label='Precisión')
ax.plot(taus_plot, rec_por_tau,  color='#E8913A', linewidth=2, label='Recall')
ax.axvline(0.50, color='black', linestyle='--', linewidth=1, alpha=0.7, label='τ=0.5 (default)')
ax.axvline(tau_youden, color='red', linestyle='--', linewidth=1.5, label=f'τ={tau_youden:.2f} (Youden)')
ax.axvline(tau_costos, color='green', linestyle='--', linewidth=1.5, label=f'τ={tau_costos:.2f} (costos)')
ax.set_xlabel('Umbral de decisión τ')
ax.set_ylabel('Valor de la métrica')
ax.set_title('Precisión y Recall vs. Umbral', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- 4. Distribución de probabilidades ---
ax = axes[1][1]
probs_fraude   = y_pred_prob[y_2023 == 1]
probs_legitimo = y_pred_prob[y_2023 == 0]
ax.hist(probs_legitimo, bins=50, alpha=0.6, color='#10B981',
        density=True, label=f'Legítimo (n={len(probs_legitimo):,})')
ax.hist(probs_fraude, bins=50, alpha=0.6, color='#DC2626',
        density=True, label=f'Fraude real (n={len(probs_fraude):,})')
ax.axvline(0.50, color='black', linestyle='--', linewidth=1.5, label='τ=0.5 (default)')
ax.axvline(tau_youden, color='orange', linestyle='--', linewidth=1.5, label=f'τ={tau_youden:.2f} (Youden)')
ax.set_xlabel('P(fraude) predicha por el modelo')
ax.set_title('Distribución de Probabilidades por Clase', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('metricas_clasificacion_fraude.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Gráfica guardada: metricas_clasificacion_fraude.png")

---
## Sección 9: Reporte completo con sklearn

In [ ]:
print("=" * 65)
print("  REPORTE COMPLETO DE CLASIFICACIÓN (sklearn)")
print("=" * 65)

print(f"\n  Umbral por defecto (τ = 0.50):")
print(classification_report(y_2023, y_pred_clase,
                             target_names=['Legítimo (0)', 'Fraude (1)']))

# Con umbral óptimo de Youden
y_pred_youden = (y_pred_prob >= tau_youden).astype(int)
print(f"\n  Con umbral óptimo de Youden (τ = {tau_youden:.3f}):")
print(classification_report(y_2023, y_pred_youden,
                             target_names=['Legítimo (0)', 'Fraude (1)']))

print(f"  AUC:  {auc:.4f}")
print(f"  Gini: {gini:.4f}  (estándar actuarial = 2×AUC−1)")

print(f"\n{'='*65}")
print(f"  RESUMEN — LO MÁS IMPORTANTE")
print(f"{'='*65}")
print(f"  VP={vp}, FN={fn}, FP={fp}, VN={vn:,}")
print(f"  Accuracy:       {(vp+vn)/n_total*100:.1f}%  ← TRAMPA: modelo trivial = {(1-y_2023.mean())*100:.1f}%")
print(f"  Precisión:      {precision*100:.1f}%  ← calidad de las alarmas generadas")
print(f"  Recall:         {recall*100:.1f}%  ← exhaustividad en detección")
print(f"  F1-Score:       {f1_manual*100:.1f}%  ← balance entre ambas")
print(f"  F2-Score:       {fbeta_score(y_2023, y_pred_clase, beta=2)*100:.1f}%  ← prioriza recall (mejor para fraude)")
print(f"  AUC:            {auc:.4f}")
print(f"  Gini:           {gini:.4f}")
print(f"  Umbral Youden:  τ = {tau_youden:.3f}")
print(f"  Umbral costos:  τ = {tau_costos:.3f}")
print(f"  Valor neto:     ${valor_neto:,.0f}")

---
## Conclusiones del Notebook 2

### Lo más importante

1. **Accuracy = trampa** cuando las clases están desbalanceadas. Con 4% de fraude, predecir siempre "legítimo" da 96% accuracy sin detectar nada.

2. **La matriz de confusión** es el punto de partida. Cada celda tiene un costo económico cuantificable.

3. **El umbral 0.5 es arbitrario.** El umbral óptimo depende de:
   - **Youden**: maximiza J = TPR − FPR 
   - **Costos del negocio**: τ* = C_FP / (C_FP + C_FN) = $3,500 / $38,500 = 0.091

4. **AUC > 0.80** (Gini > 0.60) es un buen modelo de detección de fraude.

5. **Usar F2** en fraude porque no detectar un fraude ($35,000) cuesta 10x más que una falsa alarma ($3,500).